# RAG_使用者查詢系統

# 選擇模型
請自由任意選擇下面兩個模型之一來進行範例

## Gemini

In [ ]:
from langchain_google_genai import ChatGoogleGenerativeAI

API_KEY = "這邊請改成你自己的API_KEY值"
model_name = 'gemini-2.5-flash'

llm = ChatGoogleGenerativeAI(
    model=model_name,
    google_api_key=API_KEY
)

## LM Studio 或 Ollama

In [ ]:
from langchain_openai import ChatOpenAI
model_name = 'gemma-3-12b-it'  # 指定模型名稱，模型名稱會根據下載的模型不同而改變

# 根據使用LM Studio或Ollama來選擇適當的伺服器URL
base_url = 'http://localhost:1234/v1'  # LM Studio 本地伺服器的URL
# base_url = 'http://localhost:11434/v1' # Ollama 本地伺服器的URL

llm = ChatOpenAI(
    model=model_name,
    openai_api_key="not-needed",
    openai_api_base=base_url 
)

# RAG回應使用者查詢

In [10]:
# 匯入必要模組
from langchain_chroma import Chroma  # 向量資料庫 Chroma，用來儲存與檢索文件向量
from langchain_huggingface.embeddings import HuggingFaceEmbeddings  # 文字嵌入模型
from langchain_core.messages import HumanMessage, SystemMessage  # LangChain 訊息格式，用於 LLM 對話

class QA_RAG:
    def __init__(self, llm):
        """
        初始化 QA_RAG 類別。
        參數：
            llm: 已經初始化好的大型語言模型
        """
        self.llm = llm  # 儲存 LLM 實例供後續使用

        # === 載入文字嵌入模型 ===
        # 將文字轉換成高維度向量以便進行相似度檢索。
        embeddings = HuggingFaceEmbeddings(model_name="BAAI/bge-large-zh-v1.5")

        # === 連接到現有的 Chroma 向量資料庫 ===
        # persist_directory 指向已經儲存好的資料庫資料夾
        persist_dir = "./chroma_db"

        # 連接到指定的 collection（這裡是 "clementshop_qa"）
        self.vector_store = Chroma(
            collection_name="clementshop_qa",
            embedding_function=embeddings,
            persist_directory=persist_dir
        )

    def __get_doc__(self, question):
        """
        根據使用者的問題進行相似度檢索，從向量資料庫中找出最相關的文件。
        參數：
            question: 使用者的自然語言問題
        回傳：
            最相似的文件（Document 物件列表）
        """
        return self.vector_store.similarity_search(question, k=5)  # 取最相似的5份文件

    def ask(self, question):
        """
        主函式：根據使用者的問題進行 RAG 問答。
        流程：
        1. 從資料庫中檢索最相關的 QA 文件。
        2. 將該文件內容放入 system prompt 作為規範。
        3. 呼叫 LLM 生成最終回覆。
        """
        # === 找出最相關的文件 ===
        QA = self.__get_doc__(question)

        print('找到的文件為：')
        for doc in QA:
            print(doc.page_content)
            print(doc.metadata)
            print('---------------')

        print('===============================\n')
        
        # === 構建 system prompt ===
        system_prompt = f'''
你是一位專業的電商客服代表。以下內容為公司內部的作業規範與應答準則：
{QA}

請根據上述規範，準確且禮貌地回答使用者的問題。  
請注意以下原則：
1. 回覆時不得提及或暗示任何內部文件、規章或系統資訊的存在。  
2. 僅能回應與公司平台、產品、訂單、客服流程等相關的問題。  
3. 若使用者的提問與平台或服務內容無關，請婉轉地表示無法協助，並引導回到與平台相關的主題。
'''

        # === 準備對話訊息 ===
        messages = [
            SystemMessage(system_prompt),  # 系統指令
            HumanMessage(question)         # 使用者問題
        ]

        # === 呼叫 LLM 生成回覆 ===
        response = self.llm.invoke(messages)

        # 回傳模型輸出的文字內容
        return response.content


rag = QA_RAG(llm) # 建立系統實例


In [11]:
quesion = "購物會開發票嗎？"
answer = rag.ask(quesion)
print(f'問題：{quesion}\n答案：{answer}')

找到的文件為：
Q3: 電子發票何時開立？
A3: 電子發票會於訂單完成出貨後自動開立。 若您在規定時間內未收到通知或確認信，請檢查垃圾郵件夾或聯繫客服協助處理。 若您在操作過程中遇到任何問題，建議您可先確認網路連線狀況，或重新整理頁面再試一次。 請注意，部分作業可能因例假日或物流延誤而稍有影響，敬請見諒。 我們建議您於完成操作後截圖保存，以便日後查詢紀錄或問題申訴使用。 所有個人資料均受到嚴格保護，並依據個資法相關規範進行處理。
{'source': 'qa_data\\QA_電子發票.txt'}
---------------
ClementShop 客戶服務詳細問答 - 電子發票

Q1: ClementShop 是否開立電子發票？
A1: 是的，所有訂單均開立合法電子發票。 ClementShop 的系統會即時更新相關資訊，確保您的資料與訂單狀態保持最新。 請注意，部分作業可能因例假日或物流延誤而稍有影響，敬請見諒。

Q2: 如何查詢我的電子發票？
A2: 您可於會員中心的「發票紀錄」中查詢並下載電子發票。 請注意，部分作業可能因例假日或物流延誤而稍有影響，敬請見諒。 ClementShop 的系統會即時更新相關資訊，確保您的資料與訂單狀態保持最新。
{'source': 'qa_data\\QA_電子發票.txt'}
---------------
Q6: 電子發票是否可列印？
A6: 您可自行下載 PDF 檔案後列印使用。 若申請內容涉及退款或帳號異動，處理時間將依銀行及系統作業日為準。 若您在操作過程中遇到任何問題，建議您可先確認網路連線狀況，或重新整理頁面再試一次。 請注意，部分作業可能因例假日或物流延誤而稍有影響，敬請見諒。 客服團隊提供每日 09:00 至 18:00 線上支援，將協助您解決各類問題。

Q7: 若發票資訊填錯可以修改嗎？
A7: 發票開立後不得修改，請於下單前確認資料正確。 請注意，部分作業可能因例假日或物流延誤而稍有影響，敬請見諒。 如遇特殊情況，我們將主動以電子郵件或簡訊通知您最新進度。 若您在操作過程中遇到任何問題，建議您可先確認網路連線狀況，或重新整理頁面再試一次。
{'source': 'qa_data\\QA_電子發票.txt'}
---------------
Q10: 發票會寄到我的電子郵件嗎？
A10: 發票

In [12]:
quesion = "商品寄來就壞掉了！我要退貨！"
answer = rag.ask(quesion)
print(f'問題：{quesion}\n答案：{answer}')

找到的文件為：
Q10: 哪些商品不接受退貨？
A10: 客製化、易腐壞或已拆封之商品恕不接受退貨。 所有個人資料均受到嚴格保護，並依據個資法相關規範進行處理。 若申請內容涉及退款或帳號異動，處理時間將依銀行及系統作業日為準。 客服團隊提供每日 09:00 至 18:00 線上支援，將協助您解決各類問題。 請注意，部分作業可能因例假日或物流延誤而稍有影響，敬請見諒。
{'source': 'qa_data\\QA_退貨.txt'}
---------------
Q6: 是否需寄回損壞商品？
A6: 視情況而定，客服會告知是否需寄回檢驗。 所有個人資料均受到嚴格保護，並依據個資法相關規範進行處理。 所有流程皆可於會員中心即時查詢進度，讓您清楚掌握每一步狀態。

Q7: 理賠申請可以取消嗎？
A7: 若尚未完成審核，您可提出取消申請。 客服團隊提供每日 09:00 至 18:00 線上支援，將協助您解決各類問題。 所有流程皆可於會員中心即時查詢進度，讓您清楚掌握每一步狀態。
{'source': 'qa_data\\QA_理賠.txt'}
---------------
Q4: 退貨商品需要保持什麼狀態？
A4: 商品須保持全新未使用、包裝完整並附原始發票。 我們建議您於完成操作後截圖保存，以便日後查詢紀錄或問題申訴使用。 所有流程皆可於會員中心即時查詢進度，讓您清楚掌握每一步狀態。 ClementShop 的系統會即時更新相關資訊，確保您的資料與訂單狀態保持最新。

Q5: 退貨申請審核多久？
A5: 客服將於收到申請後 3 個工作天內完成審核。 所有個人資料均受到嚴格保護，並依據個資法相關規範進行處理。 ClementShop 的系統會即時更新相關資訊，確保您的資料與訂單狀態保持最新。 若您在規定時間內未收到通知或確認信，請檢查垃圾郵件夾或聯繫客服協助處理。
{'source': 'qa_data\\QA_退貨.txt'}
---------------
Q3: 退貨需要支付運費嗎？
A3: 若為商品瑕疵或出貨錯誤，退貨運費由 ClementShop 負擔。 請注意，部分作業可能因例假日或物流延誤而稍有影響，敬請見諒。 所有個人資料均受到嚴格保護，並依據個資法相關規範進行處理。 如遇特殊情況，我們將主動以電子郵件或簡訊通知您最新進度。 若您在規定時間內未收到通知或

In [13]:
quesion = "COCO賣的比你們便宜！"
answer = rag.ask(quesion)
print(f'問題：{quesion}\n答案：{answer}')

找到的文件為：
Q7: 是否提供國際出貨？
A7: 目前 ClementShop 僅提供台灣地區配送服務。 若您在規定時間內未收到通知或確認信，請檢查垃圾郵件夾或聯繫客服協助處理。 我們建議您於完成操作後截圖保存，以便日後查詢紀錄或問題申訴使用。

Q8: 出貨時會附發票嗎？
A8: 所有訂單均為電子發票，不隨貨附紙本發票。 ClementShop 的系統會即時更新相關資訊，確保您的資料與訂單狀態保持最新。 我們建議您於完成操作後截圖保存，以便日後查詢紀錄或問題申訴使用。 所有個人資料均受到嚴格保護，並依據個資法相關規範進行處理。 若申請內容涉及退款或帳號異動，處理時間將依銀行及系統作業日為準。
{'source': 'qa_data\\QA_出貨管理與通知.txt'}
---------------
Q2: 是否可使用電話訂購？
A2: 目前僅支援線上訂購，暫不提供電話下單服務。 ClementShop 的系統會即時更新相關資訊，確保您的資料與訂單狀態保持最新。 客服團隊提供每日 09:00 至 18:00 線上支援，將協助您解決各類問題。 我們建議您於完成操作後截圖保存，以便日後查詢紀錄或問題申訴使用。 若您在規定時間內未收到通知或確認信，請檢查垃圾郵件夾或聯繫客服協助處理。 若您在操作過程中遇到任何問題，建議您可先確認網路連線狀況，或重新整理頁面再試一次。
{'source': 'qa_data\\QA_訂購方式.txt'}
---------------
Q7: 是否可以使用禮物卡或優惠券付款？
A7: 可以，您可於結帳時輸入禮物卡號或優惠代碼。 請注意，部分作業可能因例假日或物流延誤而稍有影響，敬請見諒。 如遇特殊情況，我們將主動以電子郵件或簡訊通知您最新進度。 ClementShop 的系統會即時更新相關資訊，確保您的資料與訂單狀態保持最新。 若申請內容涉及退款或帳號異動，處理時間將依銀行及系統作業日為準。 若您在規定時間內未收到通知或確認信，請檢查垃圾郵件夾或聯繫客服協助處理。
{'source': 'qa_data\\QA_付款方式.txt'}
---------------
Q3: 是否可貨到付款？
A3: 部分商品支援貨到付款，詳情請參考商品頁面標示。 客服團隊提供每日 09:00 至 18:00 線上支援，將協助您解決各類問題。 若申請內容涉及

In [14]:
quesion = "機器學習的定義"
answer = rag.ask(quesion)
print(f'問題：{quesion}\n答案：{answer}')

找到的文件為：
Q5: 忘記密碼怎麼辦？
A5: 請於登入頁面點選「忘記密碼」，系統將寄送重設密碼連結至您的電子郵件。 若您在規定時間內未收到通知或確認信，請檢查垃圾郵件夾或聯繫客服協助處理。 所有流程皆可於會員中心即時查詢進度，讓您清楚掌握每一步狀態。

Q6: 如何確認是否註冊成功？
A6: 完成註冊後，系統會自動寄出確認信至您的電子郵件信箱。 所有流程皆可於會員中心即時查詢進度，讓您清楚掌握每一步狀態。 所有個人資料均受到嚴格保護，並依據個資法相關規範進行處理。
{'source': 'qa_data\\QA_會員註冊.txt'}
---------------
Q10: 配送時會聯絡收件人嗎？
A10: 物流人員於配送前會以電話或簡訊方式通知收件人。 若您在操作過程中遇到任何問題，建議您可先確認網路連線狀況，或重新整理頁面再試一次。 請注意，部分作業可能因例假日或物流延誤而稍有影響，敬請見諒。 所有流程皆可於會員中心即時查詢進度，讓您清楚掌握每一步狀態。 客服團隊提供每日 09:00 至 18:00 線上支援，將協助您解決各類問題。
{'source': 'qa_data\\QA_運貨寄送.txt'}
---------------
Q8: 可以預訂尚未上市的商品嗎？
A8: 若商品頁面標示可預購，您可依指示完成預購流程。 我們建議您於完成操作後截圖保存，以便日後查詢紀錄或問題申訴使用。 若申請內容涉及退款或帳號異動，處理時間將依銀行及系統作業日為準。

Q9: 訂單中可以選擇不同付款方式嗎？
A9: 同一筆訂單僅能選擇一種付款方式。 如遇特殊情況，我們將主動以電子郵件或簡訊通知您最新進度。 若您在規定時間內未收到通知或確認信，請檢查垃圾郵件夾或聯繫客服協助處理。

Q10: 如何查看訂單進度？
A10: 您可登入會員中心，於「訂單查詢」中查看出貨與配送狀態。 請注意，部分作業可能因例假日或物流延誤而稍有影響，敬請見諒。 如遇特殊情況，我們將主動以電子郵件或簡訊通知您最新進度。 若您在操作過程中遇到任何問題，建議您可先確認網路連線狀況，或重新整理頁面再試一次。
{'source': 'qa_data\\QA_訂購方式.txt'}
---------------
Q6: 可以寄送至超商嗎？
A6: 部分商品支援超商取貨，您可於結帳時選擇指定門市。 所有個人資料